# Chapter 11. Deep Learning in Chemistry
## 11.2. Multilayer networks and held-out generalization

A multilayer perceptron (MLP) combines affine layers with nonlinear activation functions. We will fit a small nonlinear synthetic relationship, select a checkpoint using validation data, compare it with simpler baselines, and reload the complete inference pipeline.

### Learning objectives

- Explain why stacked affine layers alone remain affine.
- Choose an output layer and loss compatible with the target and tensor shapes.
- Separate parameter fitting, model selection, and final evaluation.
- Fit preprocessing on training data only and restore the best validation checkpoint.
- Save architecture, feature order, preprocessing, and weights together.
- Distinguish a synthetic interpolation exercise from chemical generalization.

**Run independently:** use the [course environment](Readme.md). The example uses 180 synthetic rows, one CPU thread, and at most 500 inexpensive full-batch updates with early stopping. No GPU, TensorBoard service, network, external figure, or earlier notebook output is needed.

### What changes when we add a hidden layer?

A **hidden unit** makes an intermediate feature from the inputs. An **activation** bends its response, allowing a later weighted sum to describe curved relationships. The network's **architecture** is the arrangement and sizes of these layers; training estimates the weights inside that arrangement.

**Core route:** see how hidden units combine → inspect the train/validation/test roles → follow one bounded fit → read the restored checkpoint and prediction plots. Deriving every matrix product, implementing configurable layers, and saving all checkpoint metadata are **deeper details**; their checks are useful even before you can reproduce each line.

**Research question:** a model fits the sampled range well. What evidence supports using it outside that range? Our synthetic example has a known answer, making it possible to expose an error that real experimental data might hide. The final extrapolation figure is a controlled diagnostic, not an additional round of model selection.

In [ ]:
from pathlib import Path
from time import perf_counter
from copy import deepcopy
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import torch
from torch import nn

OUTPUT_DIR = Path("outputs/chapter11_part2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 20261102
DEVICE = torch.device("cpu")
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
print(f"PyTorch {torch.__version__}; device={DEVICE}; one CPU thread")

### 11.2.1. Nonlinearity changes the function class

A two-hidden-layer MLP for regression has the form

$$
\mathbf h_1=\phi(W_1\mathbf x+\mathbf b_1),\quad
\mathbf h_2=\phi(W_2\mathbf h_1+\mathbf b_2),\quad
\hat{\mathbf y}=W_3\mathbf h_2+\mathbf b_3.
$$

The hidden activation $\phi$ supplies nonlinearity. Without it, two affine layers collapse to one:

$$
W_2(W_1\mathbf x+\mathbf b_1)+\mathbf b_2
=(W_2W_1)\mathbf x+(W_2\mathbf b_1+\mathbf b_2).
$$

Adding width or depth therefore does not automatically improve a model. It changes its capacity and optimization problem, and may increase overfitting. Scaling often helps conditioning; it does not guarantee successful optimization or accurate predictions.

In [ ]:
first = nn.Linear(3, 5).to(DEVICE)
second = nn.Linear(5, 2).to(DEVICE)
probe = torch.tensor([[0.2, 0.5, 0.7], [0.9, 0.1, 0.3]], dtype=torch.float32, device=DEVICE)
with torch.inference_mode():
    collapsed_weight = second.weight @ first.weight
    collapsed_bias = second.weight @ first.bias + second.bias
    stacked = second(first(probe))
    collapsed = probe @ collapsed_weight.T + collapsed_bias
    torch.testing.assert_close(stacked, collapsed)
print("Two affine layers exactly match one collapsed affine map, within floating-point precision.")

### Activations need not be differentiable at every point

Sigmoid and tanh are smooth but saturate at large input magnitude, where gradients become small. ReLU, $\max(0,z)$, is not differentiable at zero; automatic-differentiation libraries use a defined convention there. PyTorch uses zero for this derivative. Thus the statement that every activation must have a derivative everywhere is too strong. Backpropagation requires suitable derivative or subgradient rules for the operations used. See [PyTorch's rules for nondifferentiable functions](https://docs.pytorch.org/docs/stable/notes/autograd.html#gradients-for-non-differentiable-functions).

Our hidden layers use tanh. The final regression layer has **no activation**, allowing real-valued predictions; we do not force arbitrary regression targets into a probability range.

In [ ]:
u = torch.tensor([-1.0, 0.0, 1.0], requires_grad=True, device=DEVICE)
torch.relu(u).sum().backward()
torch.testing.assert_close(u.grad, torch.tensor([0., 0., 1.], device=DEVICE))
z = np.linspace(-4, 4, 301)
sigmoid_values = 1/(1+np.exp(-z))
activation_values = [sigmoid_values, np.tanh(z), np.maximum(0, z)]
activation_slopes = [sigmoid_values*(1-sigmoid_values), 1-np.tanh(z)**2, (z > 0).astype(float)]
fig, axes = plt.subplots(2, 3, figsize=(10, 5), layout="constrained")
for column, title in enumerate(["Sigmoid", "Tanh", "ReLU"]):
    axes[0, column].plot(z, activation_values[column], color="#287fa3")
    if column == 2:
        axes[1, column].plot(z[z < 0], activation_slopes[column][z < 0], color="#b45309")
        axes[1, column].plot(z[z > 0], activation_slopes[column][z > 0], color="#b45309")
        axes[1, column].scatter([0], [1], facecolors="white", edgecolors="#b45309")
    else:
        axes[1, column].plot(z, activation_slopes[column], color="#b45309")
    axes[0, column].set(ylabel="Activation", title=title)
    axes[1, column].set(xlabel="Input z", ylabel="Slope / autograd convention")
    for ax in axes[:, column]:
        ax.grid(alpha=0.2)
axes[1, 2].scatter([0], [0], color="#b45309", label="PyTorch convention at zero")
axes[1, 2].legend(fontsize=7)
plt.show()
print("PyTorch ReLU derivative at [-1, 0, 1]:", u.grad.tolist())

### Three simple bends can make a new shape

Here we set three ReLU units **by hand**, with bends at $x=-1,0,1$, and combine them:

$$f(x)=\operatorname{ReLU}(x+1)-2\operatorname{ReLU}(x)+\operatorname{ReLU}(x-1).$$

This is a small exact example of representation, not a trained model or a chemical law. The left panel shows the **weighted contributions**, including a negative one; the right shows their sum. The later trained model uses smooth tanh units instead.

**Predict:** calculate the sum at $x=0$ and $x=2$. Why does the negative contribution matter?

In [ ]:
hinge_x = np.linspace(-2, 2, 301)
hinge_components = np.column_stack([np.maximum(0, hinge_x+1),
                                    -2*np.maximum(0, hinge_x), np.maximum(0, hinge_x-1)])
hinge_sum = hinge_components.sum(axis=1)
np.testing.assert_allclose(hinge_sum, np.maximum(0, 1-np.abs(hinge_x)), atol=1e-14)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), layout="constrained")
for column, label in enumerate(["ReLU(x+1)", "-2 ReLU(x)", "ReLU(x-1)"]):
    axes[0].plot(hinge_x, hinge_components[:, column], label=label)
axes[0].set(xlabel="Input x", ylabel="Weighted hidden-unit contribution", title="Three reusable bends")
axes[0].legend(fontsize=8)
axes[1].plot(hinge_x, hinge_sum, color="#b45309")
axes[1].set(xlabel="Input x", ylabel="Sum of the contributions", title="One nonlinear response")
for ax in axes:
    ax.grid(alpha=0.2)
plt.show()

**Explain:** the values are 1 and 0. The negative term reduces the slope after the middle bend; the final term brings the slope back to zero. Stacking affine layers alone cannot produce this shape. In a fitted network, training selects the locations, slopes, and combinations rather than our specifying them.

### 11.2.2. Define the prediction task and splits

The known synthetic relationship is

$$
y=\sin(2x)+0.3x+\epsilon,\qquad x\sim U(-2.5,2.5),\quad
\epsilon\sim\mathcal N(0,0.1^2).
$$

Both $x$ and $y$ are dimensionless. The formula is used to generate and visualize the example, not supplied to the network as a feature. Fresh noise means a useful model need not reproduce every observation exactly.

| Split | Rows | Role |
|---|---:|---|
| Training | 108 | Fit preprocessing, weights, and baselines |
| Validation | 36 | Select the epoch and stop training |
| Test | 36 | Evaluate the fixed selected models at the end |

We fix the architecture and optimization settings before the final test comparison. Repeatedly redesigning a model after reading its test results would turn that set into another validation set. These random splits sample the same small synthetic domain; they do not test extrapolation, new scaffolds, or new laboratories.

In [ ]:
rng = np.random.default_rng(SEED)
X_raw = rng.uniform(-2.5, 2.5, size=(180, 1)).astype(np.float32)
y_raw = (np.sin(2*X_raw) + 0.3*X_raw + rng.normal(0, 0.1, size=(180, 1))).astype(np.float32)
train_index, validation_index, test_index = np.split(rng.permutation(len(X_raw)), [108, 144])
assert len(set(train_index) | set(validation_index) | set(test_index)) == 180
assert set(train_index).isdisjoint(validation_index) and set(train_index).isdisjoint(test_index)
assert set(validation_index).isdisjoint(test_index)

# Fit every preprocessing statistic on the training split alone.
x_mean = X_raw[train_index].mean(axis=0, keepdims=True)
x_scale = X_raw[train_index].std(axis=0, keepdims=True)
y_mean = y_raw[train_index].mean(axis=0, keepdims=True)
y_scale = y_raw[train_index].std(axis=0, keepdims=True)
assert np.all(x_scale > 0) and np.all(y_scale > 0)
X = torch.tensor((X_raw-x_mean)/x_scale, dtype=torch.float32, device=DEVICE)
y = torch.tensor((y_raw-y_mean)/y_scale, dtype=torch.float32, device=DEVICE)
assert X.shape == y.shape == (180, 1)
print("Training/validation/test rows:", len(train_index), len(validation_index), len(test_index))
print("Training input range:", X_raw[train_index].min(), "to", X_raw[train_index].max())

We standardize the target as well as the input using training statistics. The training loss is therefore MSE in standardized target units. We will reverse that transformation before reporting prediction errors. For multiple targets, standardizing each target changes their relative contributions to a mean loss; select target weights deliberately when units and scientific priorities differ.

### 11.2.3. Build a small configurable MLP

The main network is **1 → 16 → 16 → 1**, with tanh after each hidden layer. A batch of 108 rows has shapes `(108, 1) → (108, 16) → (108, 16) → (108, 1)`. It has 321 trainable parameters, so the existence of a good training fit would not by itself establish generalization.

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_features, hidden_sizes, out_features, activation="tanh"):
        super().__init__()
        if activation != "tanh":
            raise ValueError("This teaching architecture supports tanh hidden activations.")
        widths = [in_features, *hidden_sizes]
        layers = []
        for left, right in zip(widths[:-1], widths[1:]):
            layers.extend([nn.Linear(left, right), nn.Tanh()])
        layers.append(nn.Linear(widths[-1], out_features))
        self.layers = nn.Sequential(*layers)

    def forward(self, inputs):
        return self.layers(inputs)

ARCHITECTURE = {"in_features": 1, "hidden_sizes": [16, 16], "out_features": 1, "activation": "tanh"}
torch.manual_seed(SEED)
model = MLP(**ARCHITECTURE).to(DEVICE)
assert model(X[train_index]).shape == y[train_index].shape == (108, 1)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
assert parameter_count == 321
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.015)
print(model)
print("Trainable parameters:", parameter_count)

### 11.2.4. Train, validate, and restore the best epoch

We allow at most 500 full-batch updates and stop after 30 consecutive epochs without a lower validation loss. Validation does not contribute gradients. The test split is not read by the training loop.

Each saved best state is a **deep copy**. Merely assigning `best_state = model.state_dict()` leaves references to tensors that subsequent updates can change. At the end, restore the weights from the best validation epoch, rather than using whichever weights happened to be present when training stopped.

`model.train()` and `model.eval()` control the behavior of modules such as dropout or batch normalization. They do not themselves turn gradient recording on or off. `torch.inference_mode()` disables the recording needed for training during prediction. This tanh network has no dropout or batch normalization, but keeping the distinction explicit makes the workflow transferable. See [evaluation mode and gradient modes](https://docs.pytorch.org/docs/stable/notes/autograd.html#evaluation-mode-nn-module-eval).

In [ ]:
MAX_EPOCHS = 500
PATIENCE = 30
best_validation = float("inf")
best_epoch = 0
epochs_without_improvement = 0
best_state = None
history = []
start = perf_counter()
for epoch in range(1, MAX_EPOCHS+1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    training_prediction = model(X[train_index])
    assert training_prediction.shape == y[train_index].shape
    loss = criterion(training_prediction, y[train_index])
    assert torch.isfinite(loss)
    loss.backward()
    assert all(parameter.grad is not None and torch.isfinite(parameter.grad).all()
               for parameter in model.parameters())
    optimizer.step()
    model.eval()
    with torch.inference_mode():
        train_loss = float(criterion(model(X[train_index]), y[train_index]).item())
        validation_loss = float(criterion(model(X[validation_index]), y[validation_index]).item())
    history.append({"epoch": epoch, "train_MSE_scaled": train_loss, "validation_MSE_scaled": validation_loss})
    if validation_loss < best_validation:
        best_validation = validation_loss
        best_epoch = epoch
        best_state = deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
    if epochs_without_improvement >= PATIENCE:
        break
training_seconds = perf_counter()-start
assert best_state is not None and np.isfinite(best_validation)
model.load_state_dict(best_state)
model.eval()
with torch.inference_mode():
    restored_validation = criterion(model(X[validation_index]), y[validation_index]).item()
np.testing.assert_allclose(restored_validation, best_validation, rtol=1e-6, atol=1e-8)
history_table = pd.DataFrame(history)
reason = "validation patience reached" if epochs_without_improvement >= PATIENCE else "epoch cap reached"
print(f"Stopped at epoch {epoch}: {reason}; restored epoch {best_epoch}")
print(f"Best validation MSE in scaled target units: {best_validation:.6f}; training time: {training_seconds:.3f} s")

### 11.2.5. Freeze the decision, then evaluate once

Compare three models trained on exactly the same training rows:

- the constant training-target mean;
- a linear least-squares model;
- the selected MLP.

MAE and RMSE below are in original target units. RMSE emphasizes larger errors more strongly. A small validation set is noisy, and choosing its best epoch introduces selection effects; the untouched test set provides a separate assessment. This demonstration uses one fixed split and seed, not a broad uncertainty study.

In [ ]:
design_train = np.column_stack([np.ones(len(train_index)), X_raw[train_index]])
linear_coefficients = np.linalg.lstsq(design_train, y_raw[train_index], rcond=None)[0]
all_design = np.column_stack([np.ones(len(X_raw)), X_raw])
with torch.inference_mode():
    mlp_predictions = model(X).cpu().numpy()*y_scale+y_mean
predictions = {"training mean": np.full_like(y_raw, y_mean.item()),
               "linear least squares": all_design @ linear_coefficients,
               "MLP (validation-selected)": mlp_predictions}
metric_rows = []
for split, indices in [("training", train_index), ("validation", validation_index), ("test", test_index)]:
    for label, predicted in predictions.items():
        residual = predicted[indices]-y_raw[indices]
        assert predicted.shape == y_raw.shape and np.isfinite(residual).all()
        metric_rows.append({"split": split, "model": label,
                            "MAE": float(np.abs(residual).mean()), "RMSE": float(np.sqrt((residual**2).mean()))})
metrics = pd.DataFrame(metric_rows)
display(metrics.round(4))

In [ ]:
grid = np.linspace(float(X_raw[train_index].min()), float(X_raw[train_index].max()), 301).reshape(-1, 1)
grid_tensor = torch.tensor((grid-x_mean)/x_scale, dtype=torch.float32, device=DEVICE)
with torch.inference_mode():
    curve = model(grid_tensor).cpu().numpy()*y_scale+y_mean
linear_curve = np.column_stack([np.ones(len(grid)), grid]) @ linear_coefficients
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
axes[0].semilogy(history_table.epoch, history_table.train_MSE_scaled, label="Training")
axes[0].semilogy(history_table.epoch, history_table.validation_MSE_scaled, label="Validation")
axes[0].axvline(best_epoch, color="0.4", linestyle="--", label=f"Restored epoch {best_epoch}")
axes[0].set(xlabel="Epoch", ylabel="MSE in standardized target units", title="Validation selects the checkpoint")
for indices, label, marker, color in [(train_index, "Training rows", ".", "0.6"),
                                       (validation_index, "Validation rows", "^", "#d4953c"),
                                       (test_index, "Test rows", "x", "#406f9e")]:
    axes[1].scatter(X_raw[indices], y_raw[indices], label=label, marker=marker, s=22, color=color, alpha=0.8)
axes[1].plot(grid, np.sin(2*grid)+0.3*grid, "k--", label="Known synthetic mean")
axes[1].plot(grid, curve, color="#ba4b39", label="Selected MLP")
axes[1].plot(grid, linear_curve, color="#528044", label="Linear baseline")
axes[1].set(xlabel="Synthetic x", ylabel="Synthetic y", title="Same-domain prediction, not chemical validation")
for ax in axes:
    ax.legend(fontsize=7.5)
    ax.grid(alpha=0.2)
plt.show()

### Research diagnostic: interpolation and extrapolation are different claims

**Interpolation** predicts within the sampled input region; **extrapolation** asks about inputs beyond it. Freeze the selected model and inspect it over a wider range. The gray regions below contain no training inputs. Because the generating relationship is known in this synthetic exercise, we can compare the predictions there without pretending to have new chemical measurements.

**Predict:** does tanh's saturation make the network continue a periodic pattern automatically? Do a smooth curve and a low same-domain test error guarantee correct behavior outside the observed range?

In [ ]:
wide_x = np.linspace(-5, 5, 601).reshape(-1, 1)
wide_tensor = torch.tensor((wide_x-x_mean)/x_scale, dtype=torch.float32, device=DEVICE)
with torch.inference_mode():
    wide_prediction = model(wide_tensor).cpu().numpy()*y_scale+y_mean
known_wide_mean = np.sin(2*wide_x)+0.3*wide_x
training_low, training_high = float(X_raw[train_index].min()), float(X_raw[train_index].max())
fig, ax = plt.subplots(figsize=(9, 3.8), layout="constrained")
ax.axvspan(-5, training_low, color="0.9", label="Outside observed training range")
ax.axvspan(training_high, 5, color="0.9")
ax.plot(wide_x, known_wide_mean, "k--", label="Known synthetic mean")
ax.plot(wide_x, wide_prediction, color="#b45309", label="Frozen selected MLP")
ax.scatter(X_raw[train_index], y_raw[train_index], color="#2563eb", s=10, alpha=0.4,
            label="Training observations")
ax.set(xlabel="Synthetic x", ylabel="Synthetic y",
       title="A wider input range tests a different question")
ax.legend(fontsize=8)
plt.show()
assert np.isfinite(wide_prediction).all()

**Decision and limit:** use the trained model only within a justified domain, or obtain suitable new data for an expanded task. This figure is descriptive; we do not retune after seeing it or report it as chemical generalization. Even an in-range molecular descriptor vector can describe unfamiliar chemistry, and different assay conditions can shift the target without changing those descriptors.

**Guided exercise:** identify an interval where the frozen prediction and known mean diverge. **Selected answer:** read the actual gray-region curves; the network has no built-in periodic rule. The guarded inference helper below rejects out-of-range requests to make this limitation explicit, but its range check is not an uncertainty estimate.

The nonlinear model can follow curvature that the affine baseline cannot express. A more flexible network is not always better: data scarcity, noise, leakage, distribution shift, and optimization can outweigh extra capacity. The displayed known function is available only because these data were generated synthetically. Real molecular tasks rarely provide such a reference curve.

### 11.2.6. Multiple features and multiple targets: keep the axes

An MLP can map `(N, 3)` inputs to `(N, 2)` regression targets. `MSELoss(reduction="mean")` averages across **all $N\times2$ output elements**, not just over rows. The following four-row example checks the forward and backward tensor mechanics using two synthetic formulas. It does **not** train or evaluate a useful model on four observations.

For regression, targets should match prediction shape exactly. For binary or multilabel classification, return logits and use `BCEWithLogitsLoss` with matching floating target shapes. For mutually exclusive multiclass classification, a common setup returns `(N, C)` logits and uses `CrossEntropyLoss` with integer class indices of shape `(N,)`. Do not add sigmoid or softmax before these logits-based losses. See [PyTorch loss functions](https://docs.pytorch.org/docs/stable/nn.html#loss-functions).

In [ ]:
multi_model = MLP(in_features=3, hidden_sizes=[5, 5], out_features=2).to(DEVICE)
multi_x = torch.tensor([[0.1, 0.2, 0.3], [0.8, 0.4, 0.5], [0.3, 0.7, 0.9], [0.6, 0.1, 0.4]],
                       dtype=torch.float32, device=DEVICE)
x1, x2, x3 = multi_x.unbind(dim=1)
multi_y = torch.stack([x1.square()*x2 - 5*x2*x3 + 5, x3*x1 + 8*x2*x1 - 15], dim=1)
multi_prediction = multi_model(multi_x)
assert multi_x.shape == (4, 3) and multi_prediction.shape == multi_y.shape == (4, 2)
multi_loss = nn.MSELoss()(multi_prediction, multi_y)
torch.testing.assert_close(multi_loss, (multi_prediction-multi_y).square().sum()/multi_y.numel())
multi_loss.backward()
assert all(parameter.grad is not None and torch.isfinite(parameter.grad).all()
           for parameter in multi_model.parameters())
print("Inputs:", tuple(multi_x.shape), "targets and predictions:", tuple(multi_y.shape))
print("Output-layer weight gradient:", tuple(multi_model.layers[-1].weight.grad.shape))
print("This is a shape/gradient demonstration; the model has not been fitted.")

### 11.2.7. Save a complete inference pipeline

A prediction requires more than learned weight tensors: the architecture, feature order, input scaling, target inverse scaling, and model state must agree. The checkpoint below uses tensors and simple metadata rather than pickling a custom model or scaler object. It also records the split indices and training settings for this synthetic experiment.

We reload the file created here with `weights_only=True`, rebuild the architecture, and compare predictions. This is an **inference** checkpoint; reproducing the exact continuation of training would additionally require optimizer and random-number states. Use trusted files; restricted loading does not establish the provenance of an arbitrary checkpoint. See [PyTorch's save/load tutorial](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html).

The small inference helper accepts a two-dimensional batch and rejects values outside the observed training range. This narrow guard prevents accidental extrapolation in the exercise. Being inside that range is **not** a general applicability-domain or uncertainty guarantee.

In [ ]:
checkpoint_path = OUTPUT_DIR / "synthetic_mlp.pt"
checkpoint = {"format_version": 1, "architecture": ARCHITECTURE,
              "feature_names": ["synthetic_x"], "target_names": ["synthetic_y"], "units": "dimensionless",
              "x_mean": torch.tensor(x_mean), "x_scale": torch.tensor(x_scale),
              "y_mean": torch.tensor(y_mean), "y_scale": torch.tensor(y_scale),
              "x_training_min": torch.tensor(X_raw[train_index].min(axis=0, keepdims=True)),
              "x_training_max": torch.tensor(X_raw[train_index].max(axis=0, keepdims=True)),
              "state_dict": model.state_dict(), "best_epoch": best_epoch, "seed": SEED,
              "training_settings": {"max_epochs": MAX_EPOCHS, "patience": PATIENCE, "Adam_lr": 0.015},
              "train_index": torch.tensor(train_index), "validation_index": torch.tensor(validation_index),
              "test_index": torch.tensor(test_index), "torch_version": str(torch.__version__)}
torch.save(checkpoint, checkpoint_path)
loaded = torch.load(checkpoint_path, weights_only=True, map_location="cpu")
assert loaded["format_version"] == 1 and loaded["feature_names"] == ["synthetic_x"]
restored_model = MLP(**loaded["architecture"]).to("cpu")
restored_model.load_state_dict(loaded["state_dict"])
restored_model.eval()

def predict_raw(raw_features, fitted_model, preprocessing):
    features = torch.as_tensor(raw_features, dtype=torch.float32, device="cpu")
    if features.ndim != 2 or features.shape[1] != preprocessing["architecture"]["in_features"]:
        raise ValueError("Expected a batch with shape (number of rows, number of features).")
    if not torch.isfinite(features).all():
        raise ValueError("Features must be finite.")
    if torch.any(features < preprocessing["x_training_min"]) or torch.any(features > preprocessing["x_training_max"]):
        raise ValueError("This teaching helper restricts inference to the observed training input range.")
    fitted_model.eval()
    with torch.inference_mode():
        standardized = (features-preprocessing["x_mean"])/preprocessing["x_scale"]
        return fitted_model(standardized)*preprocessing["y_scale"]+preprocessing["y_mean"]

query = [[-2.0], [-0.5], [1.5]]
before_save = predict_raw(query, model, checkpoint)
after_reload = predict_raw(query, restored_model, loaded)
torch.testing.assert_close(before_save, after_reload, rtol=1e-6, atol=1e-7)
display(pd.DataFrame({"synthetic_x": np.array(query).ravel(), "predicted_synthetic_y": after_reload.numpy().ravel()}))
print(f"Architecture + preprocessing + weights round trip verified: {checkpoint_path}")

In [ ]:
history_table.to_csv(OUTPUT_DIR / "learning_history.csv", index=False)
metrics.to_csv(OUTPUT_DIR / "split_metrics.csv", index=False)
roles = np.full(len(X_raw), "test", dtype=object)
roles[train_index] = "training"
roles[validation_index] = "validation"
pd.DataFrame({"synthetic_x": X_raw.ravel(), "synthetic_y": y_raw.ravel(), "split": roles}).to_csv(
    OUTPUT_DIR / "synthetic_data.csv", index=False)
record = {"dataset": "synthetic sin(2*x)+0.3*x with Gaussian noise sd=0.1; no chemical measurements",
          "seed": SEED, "torch_version": str(torch.__version__), "device": "cpu", "threads": 1,
          "architecture": ARCHITECTURE, "epochs_run": len(history), "selected_epoch": best_epoch,
          "selection_metric": "validation MSE on training-standardized targets",
          "training_seconds": training_seconds, "checkpoint": checkpoint_path.name}
(OUTPUT_DIR / "experiment.json").write_text(json.dumps(record, indent=2)+"\n", encoding="utf-8")
print("Saved data/split assignments, learning history, metrics, checkpoint, and experiment metadata.")

### Exercises

1. Derive the weight and bias of two collapsed affine layers. Which operation in the MLP prevents that collapse?
2. Why can ReLU be used in backpropagation despite its kink at zero? What convention did the code check?
3. Why are the target mean and standard deviation fitted on the training split, and why must predictions be inverse-transformed before reporting RMSE?
4. Explain the separate purposes of `zero_grad`, `backward`, `step`, `eval`, and `inference_mode`.
5. Which epoch was selected? Why is the last training epoch not necessarily the one to deploy? What goes wrong with a shallow reference to `state_dict()`?
6. State the shapes for 20 samples, three features, and two regression targets. Over how many elements does default MSE average?
7. Why would predicting at $x=8$ be unsupported by this experiment? Does a low random-split test error establish performance on new molecular scaffolds?
8. Which checkpoint contents are required to predict raw-input targets correctly after restarting Python? What extra states would exact training resumption require?

<details><summary>Suggested answers</summary>

1. $W=W_2W_1$ and $b=W_2b_1+b_2$. A nonlinear hidden activation prevents a general collapse to one affine map.
2. The operation has a defined automatic-differentiation/subgradient convention at the kink; the code checked derivative zero at zero. This does not make the ordinary derivative exist there.
3. Held-out data must not influence learned preprocessing. The optimized loss is in standardized units, while scientific error reporting must use the stated original target units.
4. Clear stored gradients; compute derivatives; update parameters; select module evaluation behavior; and disable gradient recording for inference, respectively.
5. Read the recorded best epoch. Validation loss can worsen after training loss continues improving. A shallow state dictionary can keep references to changing model tensors instead of preserving the selected snapshot.
6. `(20, 3)` inputs and `(20, 2)` predictions/targets. MSE averages 40 squared residuals.
7. Training inputs lie near −2.5 to 2.5, so 8 is extrapolation. Synthetic random-split performance says nothing by itself about a chemically different test distribution.
8. Architecture, ordered feature definition, input/target preprocessing, and weights; also record version and task context. Exact continuation needs optimizer state and relevant random-number states, and may depend on the execution environment.

</details>

### From synthetic functions to chemical prediction

An MLP consumes an ordered numerical representation; its existence does not make that representation chemically adequate. Molecular descriptors, fingerprints, and graph networks encode different information. Chemical datasets also contain repeated molecules, related scaffolds, assay shifts, censored labels, and uncertain measurements. Splits and baselines must address the scientific use case. [Part 11.3](Chapter11_Part3.ipynb) examines these issues with molecular data.

### Primary documentation

- [PyTorch neural-network layers and losses](https://docs.pytorch.org/docs/stable/nn.html).
- [Autograd, nondifferentiability, and evaluation modes](https://docs.pytorch.org/docs/stable/notes/autograd.html).
- [Saving/loading models and preserving a best model state](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html).
- [PyTorch reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html): fixed seeds and deterministic operations help within this setup; exact agreement across all versions/platforms is not guaranteed.
- [scikit-learn common pitfalls: leakage and preprocessing](https://scikit-learn.org/stable/common_pitfalls.html): the same train-only preprocessing principle applies to neural networks.